In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from bayes_opt import BayesianOptimization
import warnings
import seaborn as sns
import math
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

random_state = 1
torch.manual_seed(random_state)
np.random.seed(random_state)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_state)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前使用的计算设备: {device}")


class PositionalEncoding(nn.Module):
    """位置编码模块：为序列中的每个特征 token 注入位置信息"""
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerRegressor(nn.Module):
    """基于 Transformer Encoder 的表格数据回归模型"""
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=3,
                 dim_feedforward=128, dropout=0.1, output_dim=1):
        super(TransformerRegressor, self).__init__()
        self.feature_embedding = nn.Linear(1, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=num_features + 10, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.regression_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, output_dim)
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.feature_embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        out = self.regression_head(x)
        return out


def train_transformer(model, X_train, y_train, X_val=None, y_val=None,
                      epochs=200, lr=1e-3, batch_size=64, weight_decay=1e-4,
                      patience=15, verbose=False):
    """训练 Transformer 模型，支持早停机制"""
    model.to(device)
    model.train()

    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).unsqueeze(1).to(device)
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    use_val = X_val is not None and y_val is not None
    if use_val:
        X_val_t = torch.FloatTensor(X_val).to(device)
        y_val_t = torch.FloatTensor(y_val).unsqueeze(1).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * batch_X.size(0)
        epoch_loss /= len(train_dataset)
        scheduler.step()

        if use_val:
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_t)
                val_loss = criterion(val_pred, y_val_t).item()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= patience:
                break
                
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model

def predict_transformer(model, X):
    """推理预测"""
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X).to(device)
        pred = model(X_t).cpu().numpy().flatten()
    return pred


def CV_group(data):
    """负载均衡的数据分组策略（与原版完全一致）"""
    data['ref_FO'] = data['ID'].map(str)
    ref_JQ = data['ID']
    tt_ion = pd.DataFrame(ref_JQ.value_counts(ascending=False))
    tt_ion_index = tt_ion.index
    print(tt_ion_index)

    one, two, three, four, five = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for i in range(len(tt_ion_index)):
        a = [len(one), len(two), len(three), len(four), len(five)]
        min_idx = a.index(min(a))
        target_df = pd.DataFrame(data[data['ID'] == tt_ion_index[i]])
        if min_idx == 0: one = pd.concat([one, target_df])
        elif min_idx == 1: two = pd.concat([two, target_df])
        elif min_idx == 2: three = pd.concat([three, target_df])
        elif min_idx == 3: four = pd.concat([four, target_df])
        elif min_idx == 4: five = pd.concat([five, target_df])

    print(f'one len: {len(one)} | two len: {len(two)} | three len: {len(three)} | four len: {len(four)} | five len: {len(five)}')
    return one, two, three, four, five


class ML_model():
    def __init__(self, train, test, one, two, three, four, five, features):
        self.X_train = pd.DataFrame(train)
        self.X_test = pd.DataFrame(test)
        self.y_train = train['R']
        self.y_test = test['R']

        self.X_train_1 = pd.concat([two, three, four, five], join='outer')
        self.X_test_1 = one
        self.X_train_2 = pd.concat([one, three, four, five], join='outer')
        self.X_test_2 = two
        self.X_train_3 = pd.concat([one, two, four, five], join='outer')
        self.X_test_3 = three
        self.X_train_4 = pd.concat([one, two, three, five], join='outer')
        self.X_test_4 = four
        self.X_train_5 = pd.concat([one, two, three, four], join='outer')
        self.X_test_5 = five
        self.features = features

        self.y_train_1, self.y_test_1 = self.X_train_1['R'], self.X_test_1['R']
        self.y_train_2, self.y_test_2 = self.X_train_2['R'], self.X_test_2['R']
        self.y_train_3, self.y_test_3 = self.X_train_3['R'], self.X_test_3['R']
        self.y_train_4, self.y_test_4 = self.X_train_4['R'], self.X_test_4['R']
        self.y_train_5, self.y_test_5 = self.X_train_5['R'], self.X_test_5['R']

        for i in range(1, 6):
            setattr(self, f'X_train_{i}', getattr(self, f'X_train_{i}')[self.features])
            setattr(self, f'X_test_{i}', getattr(self, f'X_test_{i}')[self.features])
        self.num_features = len(features)

    def rem_name(self, org_data):
        return org_data.loc[:, ['ref_ID']]

    def CV_train(self, model_config, X_train, y_train, X_test, y_test):
        """单折训练，内部进行标准化防止数据泄漏"""
        scaler_X = StandardScaler()
        X_tr = scaler_X.fit_transform(X_train.values)
        X_te = scaler_X.transform(X_test.values)

        scaler_y = StandardScaler()
        y_tr = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
        y_te = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

        model = TransformerRegressor(
            num_features=X_tr.shape[1],
            d_model=int(model_config['d_model']),
            nhead=int(model_config['nhead']),
            num_layers=int(model_config['num_layers']),
            dim_feedforward=int(model_config['dim_feedforward']),
            dropout=model_config['dropout'],
        )

        model = train_transformer(
            model, X_tr, y_tr,
            epochs=int(model_config['epochs']),
            lr=model_config['learning_rate'],
            batch_size=int(model_config['batch_size']),
            weight_decay=model_config['weight_decay'],
            patience=15, verbose=False
        )

        y_pred_scaled = predict_transformer(model, X_te)
        y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
        return np.sqrt(mean_squared_error(y_test.values, y_pred))

    def data_prep_for_model(self, model_config):
        rmse_1 = self.CV_train(model_config, self.X_train_1, self.y_train_1, self.X_test_1, self.y_test_1)
        rmse_2 = self.CV_train(model_config, self.X_train_2, self.y_train_2, self.X_test_2, self.y_test_2)
        rmse_3 = self.CV_train(model_config, self.X_train_3, self.y_train_3, self.X_test_3, self.y_test_3)
        rmse_4 = self.CV_train(model_config, self.X_train_4, self.y_train_4, self.X_test_4, self.y_test_4)
        rmse_5 = self.CV_train(model_config, self.X_train_5, self.y_train_5, self.X_test_5, self.y_test_5)
        
        rmse_mean = np.mean([rmse_1, rmse_2, rmse_3, rmse_4, rmse_5])
        print(f"  5-Fold Mean RMSE: {rmse_mean:.4f}")
        return -1 * rmse_mean

    def transformer_evaluate(self, d_model, nhead, num_layers, dim_feedforward,
                             dropout, learning_rate, epochs, batch_size, weight_decay):
        """贝叶斯优化目标函数"""
        d_model = int(round(d_model / 4) * 4)
        nhead = int(nhead)
        if d_model % nhead != 0:
            for nh in [nhead, nhead - 1, nhead + 1, 2, 4, 8]:
                if nh > 0 and d_model % nh == 0:
                    nhead = nh
                    break

        model_config = {
            'd_model': d_model, 'nhead': nhead, 'num_layers': int(num_layers),
            'dim_feedforward': int(dim_feedforward), 'dropout': dropout,
            'learning_rate': learning_rate, 'epochs': int(epochs),
            'batch_size': int(batch_size), 'weight_decay': weight_decay,
        }
        return self.data_prep_for_model(model_config)

    def param(self):
        """贝叶斯优化搜索 Transformer 超参数"""
        pbounds = {
            'd_model': (16, 128), 'nhead': (2, 8), 'num_layers': (1, 4),
            'dim_feedforward': (32, 256), 'dropout': (0.0, 0.4),
            'learning_rate': (1e-4, 1e-2), 'epochs': (50, 200),
            'batch_size': (16, 128), 'weight_decay': (1e-6, 1e-2),
        }
        bo = BayesianOptimization(self.transformer_evaluate, pbounds=pbounds, verbose=2, random_state=random_state)
        print('=' * 20 + '\n开始贝叶斯优化 Transformer 超参数...\n' + '=' * 20)
        bo.maximize(init_points=3, n_iter=15)
        return bo.max['params']

    def final_model(self, X_train, y_train):
        """训练最终的全局模型"""
        params = self.param()
        
        d_model = int(round(params['d_model'] / 4) * 4)
        nhead = int(params['nhead'])
        if d_model % nhead != 0:
            for nh in [nhead, nhead - 1, nhead + 1, 2, 4, 8]:
                if nh > 0 and d_model % nh == 0:
                    nhead = nh
                    break

        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
        X_train_scaled = self.scaler_X.fit_transform(X_train.values)
        y_train_scaled = self.scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()

        model = TransformerRegressor(
            num_features=X_train_scaled.shape[1],
            d_model=d_model, nhead=nhead,
            num_layers=int(params['num_layers']),
            dim_feedforward=int(params['dim_feedforward']),
            dropout=params['dropout'],
        )

        print('\n训练最终 Transformer 模型...')
        model = train_transformer(
            model, X_train_scaled, y_train_scaled,
            epochs=int(params['epochs']), lr=params['learning_rate'],
            batch_size=int(params['batch_size']), weight_decay=params['weight_decay'],
            patience=30, verbose=True
        )
        return model

    def result(self, model):
        """模型评估与可视化"""
        X_train_name = self.rem_name(self.X_train)
        X_test_name = self.rem_name(self.X_test)
        y_train = self.X_train['R']
        y_test = self.X_test['R']

        X_train_scaled = self.scaler_X.transform(self.X_train[self.features].values)
        X_test_scaled = self.scaler_X.transform(self.X_test[self.features].values)

        y_train_pred_scaled = predict_transformer(model, X_train_scaled)
        y_pred_scaled = predict_transformer(model, X_test_scaled)

        y_train_pred = self.scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).flatten()
        y_pred = self.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

        train_err = np.sqrt(mean_squared_error(y_train, y_train_pred))
        val_err = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)

        print(f"\tTraining RMSE\t\t{train_err}")
        print(f"\tTest RMSE\t\t{val_err}")
        print(f"\tMean absolute error\t{mae}")

        y_pred_df = pd.DataFrame(y_pred, columns=['pred'])
        y_test_df = pd.DataFrame(y_test.values, columns=['y_test'])
        y_train_pred_df = pd.DataFrame(y_train_pred, columns=['pred'])

        total = pd.concat([X_train_name.reset_index(drop=True), self.X_train[self.features].reset_index(drop=True),
                           y_train.reset_index(drop=True)], axis=1)
        total_train = pd.concat([total, y_train_pred_df], axis=1)
        total_train['error'] = total_train['R'] - total_train['pred']

        df_1_test = pd.concat([self.X_test[self.features].reset_index(drop=True), y_test_df], axis=1)
        total_test = pd.concat([X_test_name.reset_index(drop=True), df_1_test, y_pred_df], axis=1)
        total_test['error'] = total_test['y_test'] - total_test['pred']

        test_error = total_test.loc[:, ['y_test', 'pred', 'error']]

        plt.figure(figsize=(8, 6))
        ax = sns.scatterplot(x="y_test", y="pred", data=test_error)
        ax.set(xlim=(0, 110), ylim=(0, 105), title='Test pred-real R (Transformer)')
        ax.plot([0, 110], [0, 110], 'r--', alpha=0.6, label='Ideal y=x')
        plt.legend()
        plt.tight_layout()
        plt.show()

        return self


if __name__ == '__main__':
    df = pd.read_excel('FO TF.xlsx', sheet_name='data')
    
    df.columns = ['ID', 'MPDC', 'TMCC', 'MPDIPT',
                  'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                  'CT', 'WCA', 'WP', 'MO',
                  'DST', 'DSC', 'ST', 'CV', 'Jw', 'Js', 'R', 'ref_ID']
    
    features = ['MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                'CT', 'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', 'CV']
    
    ref_id = df['ref_ID']
    train = df[pd.to_numeric(ref_id, errors='coerce') != 1].copy()
    test = df[pd.to_numeric(ref_id, errors='coerce') == 1].copy()

    df['ref_FO'] = df['ref_ID'].map(str)
    ref_FO = df['ref_FO']
    b = pd.DataFrame(ref_FO.value_counts().sort_values(ascending=False))
    ref_FO = b.index

    df.dropna(axis=0, inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    train.dropna(axis=0, inplace=True)
    train.reset_index(drop=True, inplace=True)
    
    test.dropna(axis=0, inplace=True)
    test.reset_index(drop=True, inplace=True)

    one, two, three, four, five = CV_group(train)

    ml_model = ML_model(train, test, one, two, three, four, five, features)

    final_model = ml_model.final_model(train[features], train['R'])

    result = ml_model.result(final_model)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


X_train_scaled = ml_model.scaler_X.transform(train[features].values)
train_pred_scaled = predict_transformer(final_model, X_train_scaled)
train['pred'] = ml_model.scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

X_test_scaled = ml_model.scaler_X.transform(test[features].values)
test_pred_scaled = predict_transformer(final_model, X_test_scaled)
test['pred'] = ml_model.scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

train_pred_mae = mean_absolute_error(train['pred'], train['R'])
train_pred_rmse = np.sqrt(mean_squared_error(train['pred'], train['R']))

pred_mae = mean_absolute_error(test['pred'], test['R'])
pred_rmse = np.sqrt(mean_squared_error(test['pred'], test['R']))

print(f"MAE: {np.round(pred_mae, 2)}, RMSE: {np.round(pred_rmse, 2)}")
print(f"Train MAE: {np.round(train_pred_mae, 2)}, Train RMSE: {np.round(train_pred_rmse, 2)}")

from sklearn.metrics import r2_score

train_r2 = r2_score(train['R'], train['pred'])
train_mae = mean_absolute_error(train['R'], train['pred'])
train_mse = mean_squared_error(train['R'], train['pred'])
train_rmse = np.sqrt(train_mse)

test_r2 = r2_score(test['R'], test['pred'])
test_mae = mean_absolute_error(test['R'], test['pred'])
test_mse = mean_squared_error(test['R'], test['pred'])
test_rmse = np.sqrt(test_mse)

print("="*50)
print(f"【训练集】 R²: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
print(f"【验证集】 R²: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f}")
print("="*50)
train['mark'] = 'train'
test['mark'] = 'test'

total_data = pd.concat([train, test], ignore_index=True)

x_data = test['R'].values.reshape(-1, 1)
y_data = test['pred'].values.reshape(-1, 1)

model = LinearRegression(fit_intercept=False)
model.fit(x_data, y_data)
pred_line = model.predict(x_data)

X_title = 'Real R (%)'
Y_title = 'Pred R (%)'
title_font_size = 43
tickfont_size = 35
ticklen = 5
tickwidth = 2
title_standoff = 20
plot_width = 617
plot_height = 500

fig = px.scatter(
    total_data,
    x='R',
    y='pred',
    color='mark',
    width=plot_width,
    height=plot_height,
    range_x=[-10, 120],
    range_y=[-10, 120],
    labels={'R': X_title, 'pred': Y_title},
    color_discrete_map={'train': '1f77b4', 'test': 'ff7f0e'}
)

fig.update_layout(
    xaxis=dict(
        dtick=25,
        title_text=X_title,
        title_font={"size": title_font_size},
        tickfont=dict(family='Arial', color='black', size=tickfont_size),
        title_standoff=title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"
    ),
    yaxis=dict(
        dtick=25,
        title_text=Y_title,
        title_font={"size": title_font_size},
        tickfont=dict(family='Arial', color='black', size=tickfont_size),
        title_standoff=title_standoff,
        color='black',
        ticks="inside",
        ticklen=ticklen,
        tickwidth=tickwidth,
        title_font_family="Arial"
    ),
    plot_bgcolor='rgb(256,256,256)',
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        font=dict(size=tickfont_size - 5)
    )
)

fig.update_xaxes(showline=True, linewidth=4, linecolor='black', mirror=True)
fig.update_yaxes(showline=True, linewidth=4, linecolor='black', mirror=True)

perfect_line_x = [0, 100]
perfect_line_y = [0, 100]

fig.add_trace(go.Scatter(
    x=perfect_line_x,
    y=perfect_line_y,
    mode='lines',
    line=dict(color='red', width=2, dash='dash'),
    name='Perfect Prediction (y=x)',
    showlegend=True
))

fig.update_traces(
    marker=dict(
        size=10,
        opacity=0.65,
        line=dict(width=0.7, color='white')
    ),
    selector=dict(mode='markers')
)

fig.show()